# Axis 1 — Location, intercepts and covariate centering (`31_location_and_centering`)

**Question.** Who carries the overall level of a node, and what does an uncentered covariate do to its coefficient?

**Premises.**
1. Adding a constant $c$ to every $\theta$ of a dimension and to every $b$ of its nodes leaves every linear predictor $a(\theta - b + \dots)$ unchanged: one constraint is needed to fix the origin.
2. irtree imposes two: $\theta_p \sim N(0, \sigma^2)$ and $b_i \sim N(\mu_b = 0, \tau^2)$, with both means fixed. A fixed *population* mean does not fix the *sample* mean: the $\theta$'s and the $b$'s can each shift together, at a prior cost of about $P c^2 / (2\sigma^2)$ and $I c^2 / (2\tau^2)$.
3. The node's overall level $L = \overline{\theta} - \overline{b} + \overline{x\beta} + \overline{q\kappa}$ is identified by the likelihood. With both means fixed and no intercept, no parameter is free to carry it, so the two priors share it in proportion to $\sigma^2/P$ (traits) and $\tau^2/I$ (locations).
4. An uncentered covariate adds a constant tied to its slope: $x\beta = \bar{x}\beta + (x - \bar{x})\beta$. Whether that biases the coefficient depends on how firmly the data pin the slope: many persons for $\beta$, few items for $\kappa$.

**Predictions.**
* Part 1: with 1,000 persons and 20 items, the item locations carry about 98% of the level, the traits about 2%, and the location variance is inflated by roughly $L^2$.
* Part 2: when part of the level really comes from $\bar{x}\beta$ (the items are easy as well), centering pushes that part onto the $b$'s; uncentered, the coefficient must still compete with the $b$'s for the items' own level, so an item coefficient, pinned by few items, is pulled away even though the uncentered model is the true one.
* Part 3: when the covariates explain only deviations, uncentered $\beta$ stays close to the truth (the $b$'s absorb $\bar{x}\beta - L$), uncentered $\kappa$ is biased, and centered fits inflate the location variance.
* Part 4: an explicit intercept (a column of ones among the item covariates) gives the level its own parameter: coefficients and location variance recover whether or not the covariates are centered.

Every fit is a one-node probit Rasch tree. Each part is repeated over several simulated data sets; read the tables, not a single run.

## Setup

In [ ]:
# Install the package from this repository (editable), once per environment.
import sys, pathlib
ROOT = pathlib.Path.cwd().parents[1]
!{sys.executable} -m pip install -q -e "{ROOT}"

In [ ]:
import time, contextlib, io, warnings
import numpy as np, pandas as pd
from scipy.stats import norm
import irtree
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

RUNTIME = []
@contextlib.contextmanager
def timed(step):
    t0 = time.perf_counter(); yield; dt = time.perf_counter() - t0
    RUNTIME.append({"step": step, "seconds": round(dt, 1)})
    print(f"[{step}] {dt:.1f} s")

## Study size
`P` persons, `I` items; `SEEDS` simulated data sets per part; one chain per fit (pure Gibbs, fixed warm-up).

In [ ]:
P, I      = 1000, 20
N_ITER    = 3000
N_WARMUP  = 1000
SEEDS     = [11, 12, 13]
LEVEL     = 1.0          # node level carried by the items: mean b = -LEVEL
XBAR, XSD = 3.0, 1.0     # covariate mean and SD (deliberately far from 0)
COEF      = 0.5          # true beta and kappa
VAGUE     = irtree.NormalPrior(mean=0.0, variance=10.0)   # coefficient prior in every fit
SPEC      = irtree.IRTreeSpec(mapping=np.array([[0], [1]]))   # one binary node
print(f"P={P} I={I} N_ITER={N_ITER} seeds={SEEDS}")

## Helpers
`simulate` draws one data set; `fit` fits the one-node tree and returns the coefficients, the mean of the posterior-mean item locations and traits, and the location variance.

In [ ]:
def simulate(seed, x_const=False, q_const=False):
    """Probit Rasch node. Items easy: b ~ N(-LEVEL, 1). Covariates x (persons) and
    q (items) ~ N(XBAR, XSD). With x_const / q_const the level also includes
    XBAR*COEF from that covariate; otherwise covariates explain deviations only."""
    rng = np.random.default_rng(seed)
    theta = rng.normal(0, 1, P)
    b = rng.normal(-LEVEL, 1, I)
    x = rng.normal(XBAR, XSD, P)
    q = rng.normal(XBAR, XSD, I)
    xc = x if x_const else x - x.mean()
    qc = q if q_const else q - q.mean()
    eta = theta[:, None] - b[None, :] + COEF * xc[:, None] + COEF * qc[None, :]
    Y = (rng.random((P, I)) < norm.cdf(eta)).astype(int)
    return dict(Y=Y, x=x, q=q, b=b, theta=theta)

def fit(Y, X=None, X_names=None, Q=None, Q_names=None):
    node = irtree.ProbitRaschNode(person_covariates=True if X is not None else None,
                                  item_covariates=True if Q is not None else None)
    data = irtree.expand_data(Y, SPEC, categories_start=0, X=X, X_names=X_names, Q=Q, Q_names=Q_names)
    priors = irtree.build_default_priors(SPEC, [node])
    priors.set("beta", VAGUE); priors.set("kappa", VAGUE)
    with contextlib.redirect_stdout(io.StringIO()):
        f = irtree.irtree_fit(data, SPEC, [node], priors=priors, n_iter=N_ITER, n_warmup=N_WARMUP,
                              warmup_mode="fixed", n_chains=1, seed=1, progress=False)
    s = f.summary_df()
    out = {n: s.loc[n, "mean"] for n in s.index if n.startswith(("beta_", "kappa_"))}
    out |= {f"{n} sd": s.loc[n, "sd"] for n in s.index if n.startswith(("beta_", "kappa_"))}
    out["mean b"] = s.loc[[n for n in s.index if n.startswith("b_1_")], "mean"].mean()
    out["mean theta"] = s.loc[[n for n in s.index if n.startswith("theta_")], "mean"].mean()
    out["b.variance"] = s.loc["b.variance_1", "mean"]
    return out

def col(v):
    return v[:, None]

## Part 1 — Who carries the level when nothing else can
No covariates. The data have level $L$ = −mean(b) ≈ `LEVEL`. Prediction from premise 3: the traits take the share $(\sigma^2/P) / (\sigma^2/P + \tau^2/I)$, the items the rest.

In [ ]:
share_theta = (1 / P) / (1 / P + 1 / I)
print(f"predicted share of the level carried by the traits: {share_theta:.1%}")
rows = []
with timed("Part 1"):
    for s in SEEDS:
        d = simulate(s)
        r = fit(d["Y"])
        L = d["theta"].mean() - d["b"].mean()
        rows.append({"seed": s, "true level L": L, "true mean b": d["b"].mean(),
                     "fit mean b": r["mean b"], "fit mean theta": r["mean theta"],
                     "share carried by traits": r["mean theta"] / L,
                     "true var b": d["b"].var(ddof=1), "fit b.variance": r["b.variance"]})
part1 = pd.DataFrame(rows).round(3)
part1

**Read.** The fitted mean of the $b$'s should be close to the true one (the $b$'s carry the level), the traits' mean close to the predicted share of $L$, and the location variance inflated above the true spread because the prior $N(0, \tau^2)$ must also cover the offset of the $b$'s from 0.

## Part 2 — The level really comes from the covariate
The data are generated with the constant $\bar{x}\beta$ (or $\bar{q}\kappa$) in the predictor, so the uncentered model is the true one.

In [ ]:
rows = []
with timed("Part 2"):
    for s in SEEDS:
        d = simulate(s, x_const=True)
        for lab, x in (("uncentered", d["x"]), ("centered", d["x"] - d["x"].mean())):
            r = fit(d["Y"], X=col(x), X_names=["x"])
            rows.append({"seed": s, "covariate": "person", "coding": lab, "coef": r["beta_1_x"],
                         "coef sd": r["beta_1_x sd"], "fit mean b": r["mean b"],
                         "true mean b": d["b"].mean(), "fit b.variance": r["b.variance"],
                         "true var b": d["b"].var(ddof=1)})
        d = simulate(s, q_const=True)
        for lab, q in (("uncentered", d["q"]), ("centered", d["q"] - d["q"].mean())):
            r = fit(d["Y"], Q=col(q), Q_names=["q"])
            rows.append({"seed": s, "covariate": "item", "coding": lab, "coef": r["kappa_1_q"],
                         "coef sd": r["kappa_1_q sd"], "fit mean b": r["mean b"],
                         "true mean b": d["b"].mean(), "fit b.variance": r["b.variance"],
                         "true var b": d["b"].var(ddof=1)})
part2 = pd.DataFrame(rows).round(3)
part2

**Read.** Centered: the $b$'s must also carry $\bar{x}\beta$ = 1.5, so their mean moves by about −1.5 and the location variance inflates several-fold; the SD of $\kappa$ becomes very large. Uncentered: $\beta$ is close to 0.5, but $\kappa$ is pulled well above 0.5. The items here are also easy (mean b = −`LEVEL`), the location prior pulls the $b$'s toward 0, and the weakly pinned $\kappa$ takes up the difference through $\bar{q}\kappa$. Having the true functional form is not enough when nothing else is free to carry the level.

## Part 3 — The covariates explain only deviations
Now the level comes from the items (mean b = −`LEVEL`) and the covariates carry no constant in the truth.

In [ ]:
rows = []
with timed("Part 3"):
    for s in SEEDS:
        d = simulate(s)
        for lab, x in (("uncentered", d["x"]), ("centered", d["x"] - d["x"].mean())):
            r = fit(d["Y"], X=col(x), X_names=["x"])
            rows.append({"seed": s, "covariate": "person", "coding": lab, "coef": r["beta_1_x"],
                         "coef sd": r["beta_1_x sd"], "fit mean b": r["mean b"],
                         "true mean b": d["b"].mean(), "fit b.variance": r["b.variance"],
                         "true var b": d["b"].var(ddof=1)})
        for lab, q in (("uncentered", d["q"]), ("centered", d["q"] - d["q"].mean())):
            r = fit(d["Y"], Q=col(q), Q_names=["q"])
            rows.append({"seed": s, "covariate": "item", "coding": lab, "coef": r["kappa_1_q"],
                         "coef sd": r["kappa_1_q sd"], "fit mean b": r["mean b"],
                         "true mean b": d["b"].mean(), "fit b.variance": r["b.variance"],
                         "true var b": d["b"].var(ddof=1)})
part3 = pd.DataFrame(rows).round(3)
part3

**Read.**
* Person, uncentered: $\beta$ near 0.5; the $b$'s shift by about $\bar{x}\beta - L$ (they absorb the mismatch, premise 2).
* Item, uncentered: $\kappa$ pulled away from 0.5 with a small SD (it carries part of the level; only `I` items pin its slope).
* Centered (both): coefficients near 0.5 but the location variance inflated by about $L^2$, which also widens the SD of $\kappa$.

## Part 4 — An explicit intercept
A column of ones among the item covariates (coefficient `kappa_1_intercept`, prior N(0, 10)) is a free parameter for the level. Prediction: the location variance recovers and both coefficients recover, centered or not; centering changes only the meaning of the intercept.

In [ ]:
rows = []
ones = np.ones(I)
with timed("Part 4"):
    for s in SEEDS:
        d = simulate(s)
        for lab, x, q in (("uncentered", d["x"], d["q"]),
                          ("centered", d["x"] - d["x"].mean(), d["q"] - d["q"].mean())):
            r = fit(d["Y"], X=col(x), X_names=["x"], Q=np.column_stack([ones, q]), Q_names=["intercept", "q"])
            rows.append({"seed": s, "coding": lab, "beta": r["beta_1_x"], "beta sd": r["beta_1_x sd"],
                         "kappa": r["kappa_1_q"], "kappa sd": r["kappa_1_q sd"],
                         "intercept": r["kappa_1_intercept"], "fit mean b": r["mean b"],
                         "fit b.variance": r["b.variance"], "true var b": d["b"].var(ddof=1)})
part4 = pd.DataFrame(rows).round(3)
part4

**Read.** With the intercept: the $b$'s centered near 0, the location variance near the true spread and the same in both codings, and $\beta$, $\kappa$ the same in both codings. Centered, the intercept is about $L$; uncentered, it is about $L - \bar{x}\beta - \bar{q}\kappa$. $\kappa$ still varies across data sets with a wide SD: with 20 items, the sampled $q$ can correlate with the item locations by chance. That is sampling variability, honestly reported; the narrow SD of uncentered $\kappa$ without an intercept (Part 3) is overconfidence, not precision.

## Summary over data sets

In [ ]:
summary = pd.concat({
    "Part 2 (constant in truth)": part2.groupby(["covariate", "coding"])[["coef", "coef sd", "fit b.variance", "true var b"]].mean(),
    "Part 3 (deviations only)":   part3.groupby(["covariate", "coding"])[["coef", "coef sd", "fit b.variance", "true var b"]].mean(),
}).round(3)
summary

In [ ]:
part4.groupby("coding")[["beta", "kappa", "intercept", "fit mean b", "fit b.variance", "true var b"]].mean().round(3)

## Conclusion
* Fixing both the trait mean and the location prior mean imposes one constraint more than the location indeterminacy needs, so a node has no free intercept; its level is split between the traits and the items by their priors, mostly onto the items, inflating the location variance.
* Uncentered covariates add a constant tied to their slope. Person effects, pinned by many persons, stay close; item effects, pinned by few items, are biased and their posterior SD is too small.
* Centering alone does not give the level a home. An explicit intercept (or an estimated $\mu_b$) does, and with it centering no longer matters for the coefficients.

## Runtime

In [ ]:
rt = pd.DataFrame(RUNTIME)
rt.loc[len(rt)] = {"step": "TOTAL", "seconds": rt["seconds"].sum()}
rt